## BGL Log Feature Extraction 

We used **BGL.log** dataset and extracts features on **fixed 5-minute time windows** so the temporal structure is preserved.

It applies three substantially different feature extraction methods:
1. Manual structured window features.
2. Topic-model features with **LDA** (window documents -> topic distributions).
3. TF-IDF features built from the concatenated messages in each 5-minute window.

In [ ]:
from collections import Counter
from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.decomposition import TruncatedSVD, LatentDirichletAllocation
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_auc_score
from sklearn.pipeline import Pipeline

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 100)

WINDOW_MINUTES = 5
WINDOW_FLOOR = f'{WINDOW_MINUTES}min'

project_root = Path.cwd()
if not (project_root / 'BGL.log').exists():
    project_root = project_root.parent

log_path = project_root / 'BGL.log'
if not log_path.exists():
    raise FileNotFoundError(f'BGL.log not found from resolved root: {project_root}')

artifacts_dir = project_root / 'artifacts' / 'bgl_log_feature_extraction'
artifacts_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
# log parsing
def parse_bgl_line(line: str):
    parts = line.split(' ', 9)
    if len(parts) < 10:
        return None

    return {
        'label': parts[0],
        'unix_ts': parts[1],
        'date_token': parts[2],
        'location': parts[3],
        'timestamp_text': parts[4],
        'location_dup': parts[5],
        'facility': parts[6],
        'component': parts[7],
        'level': parts[8],
        'message': parts[9],
    }

# break log into time windows and extract features
def build_bgl_window_features(path: Path, window_minutes: int = 5):
    window_floor = f'{window_minutes}min'
    window_delta = pd.Timedelta(minutes=window_minutes)
    window_map = {}
    total_lines = 0
    parsed_lines = 0

    def init_bucket(window_start: pd.Timestamp):
        return {
            'window_start': window_start,
            'window_end': window_start + window_delta,
            'event_count': 0,
            'anomaly_count': 0,
            'message_length_sum': 0,
            'message_length_sq_sum': 0,
            'message_length_min': None,
            'message_length_max': None,
            'message_token_count_sum': 0,
            'message_token_count_sq_sum': 0,
            'message_token_count_min': None,
            'message_token_count_max': None,
            'digit_event_count': 0,
            'hex_event_count': 0,
            'cache_event_count': 0,
            'memory_event_count': 0,
            'network_event_count': 0,
            'level_counts': Counter(),
            'facility_counts': Counter(),
            'component_counts': Counter(),
            'messages': [],
            'timestamps': [],
        }

    with path.open('r', errors='ignore') as f:
        for total_lines, line in enumerate(f, start=1):
            rec = parse_bgl_line(line.rstrip('\n'))
            if rec is None:
                continue

            timestamp = pd.to_datetime(
                rec['timestamp_text'],
                format='%Y-%m-%d-%H.%M.%S.%f',
                errors='coerce',
            )
            if pd.isna(timestamp):
                continue

            parsed_lines += 1
            window_start = timestamp.floor(window_floor)
            bucket = window_map.get(window_start)
            if bucket is None:
                bucket = init_bucket(window_start)
                window_map[window_start] = bucket

            message = '' if rec['message'] is None else str(rec['message'])
            message_length = len(message)
            message_token_count = len(message.split()) if message else 0
            has_digit = int(bool(re.search(r'\d', message)))
            has_hex_like = int(bool(re.search(r'0x|[A-Fa-f0-9]{8,}', message)))
            has_cache_word = int(bool(re.search(r'cache|tlb|l1|l2', message, flags=re.IGNORECASE)))
            has_memory_word = int(bool(re.search(r'memory|parity|ecc', message, flags=re.IGNORECASE)))
            has_network_word = int(bool(re.search(r'torus|link|packet|network', message, flags=re.IGNORECASE)))

            bucket['event_count'] += 1
            bucket['anomaly_count'] += int(rec['label'] != '-')
            bucket['message_length_sum'] += message_length
            bucket['message_length_sq_sum'] += message_length ** 2
            bucket['message_token_count_sum'] += message_token_count
            bucket['message_token_count_sq_sum'] += message_token_count ** 2
            bucket['message_length_min'] = message_length if bucket['message_length_min'] is None else min(bucket['message_length_min'], message_length)
            bucket['message_length_max'] = message_length if bucket['message_length_max'] is None else max(bucket['message_length_max'], message_length)
            bucket['message_token_count_min'] = message_token_count if bucket['message_token_count_min'] is None else min(bucket['message_token_count_min'], message_token_count)
            bucket['message_token_count_max'] = message_token_count if bucket['message_token_count_max'] is None else max(bucket['message_token_count_max'], message_token_count)
            bucket['digit_event_count'] += has_digit
            bucket['hex_event_count'] += has_hex_like
            bucket['cache_event_count'] += has_cache_word
            bucket['memory_event_count'] += has_memory_word
            bucket['network_event_count'] += has_network_word
            bucket['level_counts'][rec['level']] += 1
            bucket['facility_counts'][rec['facility']] += 1
            bucket['component_counts'][rec['component']] += 1
            bucket['messages'].append(message)
            bucket['timestamps'].append(timestamp)

    records = []
    for window_index, window_start in enumerate(sorted(window_map.keys())):
        bucket = window_map[window_start]
        event_count = bucket['event_count']
        anomaly_count = bucket['anomaly_count']
        level_counts = bucket['level_counts']
        facility_counts = bucket['facility_counts']
        component_counts = bucket['component_counts']
        timestamps = bucket['timestamps']

        message_length_mean = bucket['message_length_sum'] / event_count if event_count else 0.0
        message_token_count_mean = bucket['message_token_count_sum'] / event_count if event_count else 0.0
        message_length_var = max(bucket['message_length_sq_sum'] / event_count - message_length_mean ** 2, 0.0) if event_count else 0.0
        message_token_count_var = max(bucket['message_token_count_sq_sum'] / event_count - message_token_count_mean ** 2, 0.0) if event_count else 0.0
        gap_mean_seconds = 0.0
        gap_std_seconds = 0.0
        if len(timestamps) > 1:
            gaps = pd.Series(sorted(timestamps)).diff().dt.total_seconds().dropna()
            if not gaps.empty:
                gap_mean_seconds = float(gaps.mean())
                gap_std_seconds = float(gaps.std(ddof=0)) if len(gaps) > 1 else 0.0

        dominant_level_count = level_counts.most_common(1)[0][1] if level_counts else 0
        dominant_facility_count = facility_counts.most_common(1)[0][1] if facility_counts else 0
        dominant_component_count = component_counts.most_common(1)[0][1] if component_counts else 0

        records.append({
            'window_index': window_index,
            'window_start': window_start,
            'window_end': bucket['window_end'],
            'window_minutes': window_minutes,
            'event_count': event_count,
            'anomaly_count': anomaly_count,
            'is_anomaly': int(anomaly_count > 0),
            'anomaly_rate': anomaly_count / event_count if event_count else 0.0,
            'message_length_mean': message_length_mean,
            'message_length_std': float(np.sqrt(message_length_var)),
            'message_length_min': int(bucket['message_length_min']) if bucket['message_length_min'] is not None else 0,
            'message_length_max': int(bucket['message_length_max']) if bucket['message_length_max'] is not None else 0,
            'message_token_count_mean': message_token_count_mean,
            'message_token_count_std': float(np.sqrt(message_token_count_var)),
            'message_token_count_min': int(bucket['message_token_count_min']) if bucket['message_token_count_min'] is not None else 0,
            'message_token_count_max': int(bucket['message_token_count_max']) if bucket['message_token_count_max'] is not None else 0,
            'digit_event_rate': bucket['digit_event_count'] / event_count if event_count else 0.0,
            'hex_event_rate': bucket['hex_event_count'] / event_count if event_count else 0.0,
            'cache_event_rate': bucket['cache_event_count'] / event_count if event_count else 0.0,
            'memory_event_rate': bucket['memory_event_count'] / event_count if event_count else 0.0,
            'network_event_rate': bucket['network_event_count'] / event_count if event_count else 0.0,
            'unique_level_count': len(level_counts),
            'dominant_level_share': dominant_level_count / event_count if event_count else 0.0,
            'unique_facility_count': len(facility_counts),
            'dominant_facility_share': dominant_facility_count / event_count if event_count else 0.0,
            'unique_component_count': len(component_counts),
            'dominant_component_share': dominant_component_count / event_count if event_count else 0.0,
            'mean_interarrival_seconds': gap_mean_seconds,
            'std_interarrival_seconds': gap_std_seconds,
            'window_text': ' '.join(bucket['messages']),
            'timestamp_count': len(timestamps),
        })

    window_df = pd.DataFrame(records).sort_values('window_start').reset_index(drop=True)
    return window_df, total_lines, parsed_lines

In [ ]:
window_df, total_lines_scanned, parsed_lines = build_bgl_window_features(log_path, window_minutes=WINDOW_MINUTES)

print(f'Total lines scanned: {total_lines_scanned:,}')
print(f'Parsed log events: {parsed_lines:,}')
print(f'5-minute windows created: {len(window_df):,}')
print('Window label distribution (0=normal,1=anomaly):')
print(window_df['is_anomaly'].value_counts(normalize=True).rename('ratio'))
print('Window span:')
print(window_df[['window_start', 'window_end']].head(3))

display(window_df.head(10))

## Manual Structured Feature Extraction
Manual features are computed **per 5-minute window** using lexical, temporal, and system-pattern indicators aggregated from the events inside each window.

In [ ]:
manual_feature_cols = [
    'window_index',
    'window_minutes',
    'event_count',
    'anomaly_count',
    'anomaly_rate',
    'message_length_mean',
    'message_length_std',
    'message_length_min',
    'message_length_max',
    'message_token_count_mean',
    'message_token_count_std',
    'message_token_count_min',
    'message_token_count_max',
    'digit_event_rate',
    'hex_event_rate',
    'cache_event_rate',
    'memory_event_rate',
    'network_event_rate',
    'unique_level_count',
    'dominant_level_share',
    'unique_facility_count',
    'dominant_facility_share',
    'unique_component_count',
    'dominant_component_share',
    'mean_interarrival_seconds',
    'std_interarrival_seconds',
]

window_manual_features = window_df[['window_start', 'window_end', 'is_anomaly'] + manual_feature_cols].copy()

display(window_manual_features.head(10))
print('Manual window-feature columns:', len(manual_feature_cols))

In [ ]:
# Visualiz using 5-minute window for num of events and anomaly rate over time
viz_df = window_df[['window_start', 'event_count', 'anomaly_rate', 'is_anomaly']].copy()

fig, ax1 = plt.subplots(figsize=(16, 8))
line1, = ax1.plot(viz_df['window_start'], viz_df['event_count'], color='steelblue', linewidth=0.8, alpha=0.8, label='Event count')
ax1.set_title('BGL 5-Minute Windows: Event Count and Anomaly Rate Over Time')
ax1.set_xlabel('Window start')
ax1.set_ylabel('Event count', color='steelblue')
ax1.tick_params(axis='y', labelcolor='steelblue')

ax2 = ax1.twinx()
line2, = ax2.plot(viz_df['window_start'], viz_df['anomaly_rate'], color='darkorange', linewidth=1.2, alpha=0.9, label='Anomaly rate')
scatter = ax2.scatter(
    viz_df.loc[viz_df['is_anomaly'] == 1, 'window_start'],
    viz_df.loc[viz_df['is_anomaly'] == 1, 'anomaly_rate'],
    color='crimson',
    s=10,
    alpha=0.45,
    label='Anomalous windows',
)
ax2.set_ylabel('Anomaly rate', color='darkorange')
ax2.tick_params(axis='y', labelcolor='darkorange')

handles = [line1, line2, scatter]
labels = [h.get_label() for h in handles]
legend = ax1.legend(
    handles,
    labels,
    loc='lower center',
    bbox_to_anchor=(0.5, 1.02),
    ncol=3,
    frameon=True,
    framealpha=0.95,
)
legend.set_zorder(1000)

fig.tight_layout(rect=[0, 0, 1, 0.92])
manual_viz_path = artifacts_dir / 'bgl_manual_window_features_over_time.png'
plt.savefig(manual_viz_path, dpi=180, bbox_inches='tight')
plt.show()

## LDA Topic Feature Extraction
Each 5-minute window is treated as a document and transformed into topic-distribution features using LDA.

In [ ]:
lda_docs = window_df['window_text'].fillna('')
count_vectorizer = CountVectorizer(max_features=2500, min_df=3, ngram_range=(1, 2), stop_words='english')
X_counts = count_vectorizer.fit_transform(lda_docs)

# set total number of topics to extract 
n_topics = 8
lda_model = LatentDirichletAllocation(n_components=n_topics, random_state=42, learning_method='batch')
X_topics = lda_model.fit_transform(X_counts)

topic_cols = [f'lda_topic_{i}' for i in range(n_topics)]
topic_df = pd.DataFrame(X_topics, columns=topic_cols, index=window_df.index)
window_df = pd.concat([window_df, topic_df], axis=1)
window_df['dominant_topic'] = topic_df.to_numpy().argmax(axis=1)

vocab = np.array(count_vectorizer.get_feature_names_out())
top_k = 10
topic_keywords = {}
for i, topic_weights in enumerate(lda_model.components_):
    top_idx = np.argsort(topic_weights)[-top_k:][::-1]
    topic_keywords[f'topic_{i}'] = ', '.join(vocab[top_idx])

topic_keywords_df = pd.DataFrame(
    [{'topic': key, 'top_terms': value} for key, value in topic_keywords.items()]
)

lda_feature_cols = ['window_start', 'window_end', 'is_anomaly', 'dominant_topic'] + topic_cols
lda_window_features = window_df[lda_feature_cols].copy()

display(topic_keywords_df)
display(lda_window_features.head(10))
print('LDA feature columns:', len(topic_cols))

## TF-IDF 
Each 5-minute window is treated as one document by concatenating the messages inside that window, then TF-IDF and SVD are applied for visualization and downstream classification.

In [ ]:
text_for_viz = window_df['window_text'].fillna('')
tfidf_viz = TfidfVectorizer(max_features=1500, min_df=3, ngram_range=(1, 2), stop_words='english')
X_viz = tfidf_viz.fit_transform(text_for_viz)

# SVD for 2D visualization
svd = TruncatedSVD(n_components=2, random_state=42)
X_2d = svd.fit_transform(X_viz)

plot2_df = pd.DataFrame({
    'x': X_2d[:, 0],
    'y': X_2d[:, 1],
    'is_anomaly': window_df['is_anomaly'].to_numpy(),
    'window_start': window_df['window_start'].to_numpy(),
})
step = max(1, len(plot2_df) // 50000)
plot2_df = plot2_df.iloc[::step].copy()

plt.figure(figsize=(10, 7))
scatter = plt.scatter(
    plot2_df['x'],
    plot2_df['y'],
    c=plot2_df['is_anomaly'],
    cmap='coolwarm',
    s=9,
    alpha=0.35,
)
plt.colorbar(scatter, label='is_anomaly')
plt.title('TF-IDF + SVD Window Text Embedding (BGL, 5-Minute Windows)')
plt.xlabel('SVD-1')
plt.ylabel('SVD-2')
plt.tight_layout()

tfidf_viz_path = artifacts_dir / 'bgl_window_tfidf_svd_density.png'
plt.savefig(tfidf_viz_path, dpi=180, bbox_inches='tight')
plt.show()

In [ ]:
# using TF-IDF features on 5-minute windows
clf_df = window_df[['window_start', 'window_text', 'is_anomaly']].dropna().sort_values('window_start').reset_index(drop=True)
if len(clf_df) < 10:
    raise ValueError('Not enough 5-minute windows for classification.')

split_idx = int(len(clf_df) * 0.8)
split_idx = min(max(split_idx, 1), len(clf_df) - 1)
train_df = clf_df.iloc[:split_idx].copy()
test_df = clf_df.iloc[split_idx:].copy()

clf_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, min_df=3, ngram_range=(1, 2), stop_words='english')),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced')),
])

clf_pipeline.fit(train_df['window_text'], train_df['is_anomaly'])
pred = clf_pipeline.predict(test_df['window_text'])
proba = clf_pipeline.predict_proba(test_df['window_text'])[:, 1]

print('Train windows:', len(train_df))
print('Test windows:', len(test_df))
print('F1 score:', f1_score(test_df['is_anomaly'], pred))
if test_df['is_anomaly'].nunique() > 1:
    print('ROC-AUC:', roc_auc_score(test_df['is_anomaly'], proba))
else:
    print('ROC-AUC: skipped because the test split contains one class only')
print('Confusion matrix:\n', confusion_matrix(test_df['is_anomaly'], pred))
print('Classification report:\n', classification_report(test_df['is_anomaly'], pred, digits=4))

In [ ]:
# export final datasets and features
window_features = window_df.drop(columns=['window_text']).copy()
window_text_docs = window_df[['window_start', 'window_end', 'is_anomaly', 'window_text']].copy()

features_path = artifacts_dir / 'bgl_window_features_5min.csv'
text_docs_path = artifacts_dir / 'bgl_window_text_5min.csv'
lda_topics_path = artifacts_dir / 'bgl_lda_topics.csv'
sample_path = artifacts_dir / 'bgl_window_sample.csv'

window_features.to_csv(features_path, index=False)
window_text_docs.to_csv(text_docs_path, index=False)
if 'topic_keywords_df' in globals():
    topic_keywords_df.to_csv(lda_topics_path, index=False)
window_df.head(1000).to_csv(sample_path, index=False)

print('Final windows:', len(window_features))
print('Anomalous windows:', int(window_features['is_anomaly'].sum()))